In [11]:
import sys
from pathlib import Path

CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

DATASET_FILE = CWD / "result.json"
# OUT_CHECKPOINT_FILE = CWD / "leanrag_checkpoint.json"
# USE_CHECKPOINT_AS_CACHE = True # prefer data in the checkpoint file over re-computing?

secrets = CWD / "secrets.env"

if not secrets.is_file():
    raise ValueError(f"secrets file at '{secrets}' does not exist")

from dotenv import load_dotenv
load_dotenv(secrets)

# (START WITH G0 IN MEMGRAPH) ...

True

In [12]:
# define batch embed functions
import asyncio
from utils import batched
import litellm
from litellm.exceptions import APIError, RateLimitError, APIResponseValidationError
from tqdm import tqdm
from utils.models import AsyncList, AsyncProgressBar, EntityDescEmbed, Entity
import json

EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"
ENTITY_BATCH_SIZE = 32
MAX_PARALLEL_EMBED = 8

USE_EMBED_CACHE = True # whether to pull from cache or recompute
embed_cache_file = CWD / "embed_cache.json"

async def batch_embed_descriptions(batch:list[Entity], acc:AsyncList, embed_sem:asyncio.Semaphore, pbar:AsyncProgressBar) -> None:
    """ embed a single batch of entity descriptions """
    async with embed_sem:
        resp = await litellm.aembedding(model=EMBED_MODEL, input=[e['desc'] for e in batch])
    batch_embed = resp['data']
    # unpack batch to entity_key -> description pairs
    rows = [
        EntityDescEmbed(key=batch[emb.index]["key"],desc_embed=emb.embedding)
        for emb in sorted(batch_embed, key=lambda e: e.index)
    ]
    await acc.extend(rows)
    await pbar.update(len(batch))

async def embed_all_entity_descriptions(entities:list[Entity], batch_size:int, max_parallel:int, pbar:AsyncProgressBar ) -> AsyncList[EntityDescEmbed]:
    """
    embed all entity descriptions in batches
    """
    
    acc = AsyncList()

    if USE_EMBED_CACHE and embed_cache_file.is_file():
        with open(embed_cache_file, "r") as cf:
            data = json.load(cf)
        await acc.extend(data)
        await pbar.update(len(entities))
        return acc

    embed_sem = asyncio.Semaphore(max_parallel)

    batch_embed_tasks=[]
    for batch in batched(entities, batch_size):
        batch_embed_tasks.append(batch_embed_descriptions(batch, acc, embed_sem, pbar))

    results = await asyncio.gather(*batch_embed_tasks, return_exceptions=True)

    # cache acc
    with open(embed_cache_file, "w") as cf:
        json.dump(acc.get_list(),cf)

    print(f"created embeddings for entity descriptions for #{len(entities)} entities")
    print(f"results:\n{results}")


    return acc

In [13]:
# Individual component testing -- outside agg layer func:
from utils import mg_driver
from utils.models import Finding
from math import log

await mg_driver.init()

CLUSTER_SIZE = 20
type AggEntityKey = str
findings_map : dict[AggEntityKey, list[Finding]] = {} # todo: create AsyncDict wrapper
fmap_lock = asyncio.Lock()

total_entities = await mg_driver.count_entities()
max_depth = round(log(total_entities, CLUSTER_SIZE)) + 1 # from LeanRAG
print(f"max depth: {max_depth}")

max depth: 3


In [41]:
import importlib
importlib.reload(mg_driver)
await mg_driver.init()

In [ ]:
# Individual component testing -- INSIDE agg layer func:
from utils.models import Entity
from math import ceil

layer = 1
# 1. collect all entities in the layer
entities:list[Entity] = await mg_driver.get_entities_for_layer(layer) # list of 620 entities
n_entities:int = len(entities) #
n_components:int = ceil(n_entities/CLUSTER_SIZE)


# calculate the BIC recommendation of n_components. Pick maximum from BIC recc or heuristic



print(f"{n_entities} entities. calculated {n_components} components for this layer")

24 entities. calculated 2 components for this layer


In [43]:
# stop recursion in 3 cases. return the 'root' entities
if layer > max_depth or n_entities <= 2 or n_components <= 4: raise StopIteration()

StopIteration: 

In [16]:
# 2. batch embed them --- or use cached embeddings
pbar = AsyncProgressBar(total=n_entities, desc="Entity description embeddings")
entity_desc_embeds:AsyncList[EntityDescEmbed] = await embed_all_entity_descriptions(entities, ENTITY_BATCH_SIZE, MAX_PARALLEL_EMBED, pbar=pbar)
await pbar.close()

if len(entity_desc_embeds)>0:
    print(f"created {len(entity_desc_embeds)} embeddings.\nelement 0: {entity_desc_embeds[0]}")

Entity description embeddings: 100%|██████████| 620/620 [00:00<00:00, 4009.11it/s]

created 620 embeddings.
element 0: {'key': 'strategic', 'desc_embed': [0.05259411782026291, -0.029194263741374016, 0.007344285026192665, 0.011979821138083935, 0.011564440093934536, -0.0680566355586052, 0.023242680355906487, -0.005486105103045702, -0.009131170809268951, -0.03589226305484772, -0.0166719201952219, 0.05276946723461151, 0.059050217270851135, -0.030105864629149437, 0.01756897009909153, 0.003416660474613309, -0.015059072524309158, 0.029149765148758888, 0.0024266825057566166, 0.0014849668368697166, -0.0025521102361381054, -0.044214800000190735, 0.027607262134552002, -0.03362711891531944, -0.013145661912858486, 0.03960601985454559, 0.02093149907886982, 0.03010532818734646, 0.05704803392291069, -0.03319688141345978, -0.005177890881896019, -0.013221214525401592, 0.00014617544366046786, 0.026356859132647514, 0.021011441946029663, -0.01997324824333191, -0.04156750813126564, 0.03089817985892296, -0.03719869256019592, -0.021031877025961876, 0.04845864698290825, 0.027952924370765686, 

In [17]:
from sklearn.mixture import GaussianMixture
import numpy as np
from utils.models import Cluster
# 3. Feed embeds into GMM to partition into clusters
X = np.asarray([e["desc_embed"] for e in entity_desc_embeds], dtype=np.float32)
gmm:GaussianMixture = GaussianMixture(
    n_components= n_components,
    covariance_type="diag",
    random_state=0,
    reg_covar=1e-6,
    max_iter=300,
    n_init=3
)
gmm.fit(X)
responsibilities = gmm.predict_proba(X) # probability that embedding i belongs to component k
labels = responsibilities.argmax(axis=1) # most likely component for embedding
# cluster using hard labels
# maps cluster # -> list of entities
clusters:dict[int, Cluster] = {k:[] for k in range(n_components)}
for i, k in enumerate(labels):
    clusters[int(k)].append(entities[i])

In [18]:
# peek at cluster 0
print(f"cluster 0 ({len(clusters[0])} entities)")
clusters[0]

cluster 0 (11 entities)


[{'key': 'delta_yaw',
  'name': 'delta_yaw',
  'desc': "Delta_yaw is a feature within the dataset that captures the incremental changes in the player's camera rotation around the vertical axis between consecutive samples. It is part of the aiming information collected from Minecraft players, both with and without an aimbot enabled.",
  'degree': 2},
 {'key': 'gru',
  'name': 'gru',
  'desc': 'Gated Recurrent Unit (GRU) is a simpler, LSTM-like recurrent neural network designed to tackle the vanishing/exploding gradient problem in traditional RNNs by omitting the output gate.',
  'degree': 3},
 {'key': 'full_state_at_each_time_step',
  'name': 'full_state_at_each_time_step',
  'desc': '"Full state at each time step" describes the complete internal representation or memory state of the GRU network that is made available at every step of the sequence processing. This means that all information processed up to that point is exposed without any gating mechanism to control it.',
  'degree': 2

In [19]:
from typing import Coroutine, Any
from utils.models import AggEntity
from utils import signatures

# TODO: consider using this semaphore (add it as a parameter to the generate_aggregate_node/rel calls)
# MAX_CONCURRENT_REQUESTS = 32
# agg_llm_sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

async def build_aggregate_entity_func(cluster: Cluster) -> tuple[AggEntity | None, list[Finding] | None]:
    input_rows: list[str] = []
    input_rows.append("ENTITIES: entity_name, entity_description, entity_degree")
    for i, entity in enumerate(cluster):
        input_rows.append(f"{i}: {entity['name']}, {entity['desc']}, {entity['degree']}")
    input_rows.append("")

    intra_rels = await mg_driver.get_intra_cluster_relations(cluster)
    input_rows.append("RELATIONS: source_entity, target_entity, relation_description")
    for i, rel in enumerate(intra_rels):
        input_rows.append(f"{i}: {rel['source_entity']}, {rel['target_entity']}, {rel['relation_description']}")

    input_text = "\n".join(input_rows)
    return await signatures.generate_aggregate_node(input_text=input_text)

# test out the builder
# _findings_map_test : dict[AggEntityKey, list[Finding]] = {}
# _agg_test = await build_aggregate_entity(clusters[0], _findings_map_test)

In [20]:
# creates aggregate nodes, 
aggregates:AsyncList[AggEntity] = AsyncList()
cluster_backref:dict[AggEntityKey, Cluster] = {}
cb_lock = asyncio.Lock()
aggregation_tasks = []

async def _aggregate_task(cluster: Cluster):
    """
    create new aggregate entity from cluster
    - store it in 'aggregates' list.
    - map new agg entity -> findings list in the findings map
    - map new agg entity -> child entity list (cluster) in cluster backref
    """
    new_parent, new_findings = await build_aggregate_entity_func(cluster)
    if not (new_parent and new_findings):
        print("ERROR! Got none for new parent or new findings")
        await pbar.update(1)
        return

    key: AggEntityKey = new_parent["key"]

    # update both structures atomically in a fixed lock order
    async with fmap_lock:
        async with cb_lock:
            if key in findings_map:
                # duplicate: merge into original
                findings_map[key].extend(new_findings)
                cluster_backref[key].extend(cluster)
                is_dup = True
            else:
                # first occurrence
                findings_map[key] = list(new_findings)
                cluster_backref[key] = list(cluster)
                is_dup = False

    # only keep one aggregate node per key
    if not is_dup:
        await aggregates.append(new_parent)

    await pbar.update(1)

aggregation_tasks = []
for _, cluster in clusters.items():
    aggregation_tasks.append(_aggregate_task(cluster))

from utils.models import AsyncProgressBar
pbar = AsyncProgressBar(total=len(aggregation_tasks), desc="agg tasks (layer 0)")

# clear shared state safely
async with fmap_lock:
    findings_map.clear()
await aggregates.clear()
async with cb_lock:
    cluster_backref.clear()

results=await asyncio.gather(*aggregation_tasks, return_exceptions=True)
print(results)
print(f"final # aggregates: {len(aggregates)}")

agg tasks (layer 0):  42%|████▏     | 13/31 [00:00<00:00, 128.70it/s]

[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
final # aggregates: 24


In [21]:
# (testing) double check no duplicate aggregate entities
_a_keys = [e['key'] for e in aggregates]
if(len(_a_keys) != len(set(_a_keys))):
    print("at least 1 duplicate key found")

In [ ]:
# insert in memgraph
for agg in aggregates:
    await mg_driver.create_aggregate_entity(agg, cluster_backref[agg['key']], layer+1)

In [37]:
import importlib
importlib.reload(mg_driver)
await mg_driver.init()

In [38]:
from itertools import combinations
from utils import Tokenizer
from utils.models import IntrClusterRel

def icr_desc_fallback_concat(inter_cluster_relations:list[IntrClusterRel]) -> str:
    """ LeanRAG F(rel) when connectivity strength is below threshold (tau): use simple concat of inter-cluster-relations """
    # (format from LeanRAG)
    return "\n".join([f"relationship<|>{r["source_entity"]}<|>{r["target_entity"]}<|>{r["relation_description"]}" for r in inter_cluster_relations])

allowed_tokens = ( max_depth - layer ) * 40 * 2 # (TAU) from LeanRAG

x =0
for aggJ, aggK in combinations(aggregates, 2):
    # collect all inter-cluster relations between entities in cj and entities in ck
    inter_cluster_rel:list[IntrClusterRel] = await mg_driver.get_inter_cluster_relations(cluster_backref[aggJ['key']],cluster_backref[aggK['key']])
    if len(inter_cluster_rel) <= 0:
        # not well-supported, so we don't need to create a relation between the clusters.
        continue

    # print(f"{aggJ['key']}-{aggK['key']}: {inter_cluster_rel}")
    # cumulative number of tokens for all inter-cluster relation descriptions.. LeanRAG 'connectivity strength'
    n_tokens_intercluster_rel = sum([len(Tokenizer.encode(r['relation_description'])) for r in inter_cluster_rel])
    # print(f"{aggJ['key']}-{aggK['key']}: {n_tokens_intercluster_rel}")
    if n_tokens_intercluster_rel > allowed_tokens:
        icr_desc:str = await signatures.generate_aggregate_rel_desc(aggJ, aggK, inter_cluster_rel)
    else:
        icr_desc:str = icr_desc_fallback_concat(inter_cluster_rel)

    await mg_driver.create_inter_cluster_relation(aggJ, aggK, icr_desc, layer+1)
    x+=1

print(f"created {x} inter-aggregate relations")

created 102 inter-aggregate relations


In [ ]:
#TODO: add entity_type during extraction
raise RuntimeError("not ready to run this cell yet")
from math import log, round, ceil
from sklearn.mixture import GaussianMixture
from utils import signatures
from utils.models import AggEntity, Finding, Entity, Cluster, IntrClusterRel
from utils import Tokenizer, mg_driver
from itertools import combinations
import numpy as np

CLUSTER_SIZE = 5 # hyperparameter , soft

type AggEntityKey = str
findings_map : dict[AggEntityKey, list[Finding]] = {}

async def build_aggregate_entity(cluster:Cluster, findings_map:dict[AggEntityKey, list[Finding]]) -> AggEntity:
    """creates parent node for a cluster. stores findings outside of graph in findings map"""
    
    # From LeanRAG: assemble string of 2 CSV-styled blocks. 1st the entities, then the relations between them
    input_rows:list[str] = []
    input_rows.append ("ENTITIES: entity_name, entity_description, entity_degree") # header
    for i, entity in enumerate(cluster): #TODO: limit to top-n , sort by degree descending (check if LeanRAG does this)
        input_rows.append(f"{i}: {entity['name']}, {entity['desc']}, {entity['degree']}")
    input_rows.append("")
    
    intra_rels = await mg_driver.get_intra_cluster_relations(cluster)
    input_rows.append ("RELATIONS: source_entity, target_entity, relation_description") # header
    for i, rel in enumerate(intra_rels):
        input_rows.append(f"{i}: {rel['source_entity']}, {rel['target_entity']}, {rel['relation_description']}")
    
    input_text = "\n".join(input_rows)
    agg_entity:AggEntity = await signatures.generate_aggregate_node(input_text=input_text, findings_map=findings_map) #TODO: check for malformed output, key uniqueness

    return agg_entity


def icr_desc_fallback_concat(inter_cluster_relations:list[IntrClusterRel]) -> str:
    """ LeanRAG F(rel) when connectivity strength is below threshold (tau): use simple concat of inter-cluster-relations """
    # (format from LeanRAG)
    return "\n".join([f"relationship<|>{r["source_entity"]}<|>{r["target_entity"]}<|>{r["relation_description"]}" for r in inter_cluster_relations])

async def aggregate_layer(layer:int)->list[Entity]:
    """
    recursively aggregates the layer.
    Finally returns the entities in the 'root' layer.
    """
    global max_depth

    # 1. collect all entities in the layer
    entities:list[Entity] = await mg_driver.get_entities_for_layer(layer)
    n_entities:int = len(entities)
    n_components:int = ceil(n_entities/CLUSTER_SIZE)

    # stop recursion in 3 cases. return the 'root' entities
    if layer > max_depth or n_entities <= 2 or n_components <= 4: return entities
    
    # 2. batch embed them
    entity_desc_embeds:AsyncList[EntityDescEmbed] = await embed_all_entity_descriptions([e['desc'] for e in entities], ENTITY_BATCH_SIZE, MAX_PARALLEL_EMBED)
    
    # 3. Feed embeds into GMM to partition into clusters
    X = np.asarray([e["desc_embed"] for e in entity_desc_embeds], dtype=np.float32)
    gmm:GaussianMixture = GaussianMixture(
        n_components= n_components,
        covariance_type="diag",
        random_state=0,
        reg_covar=1e-6,
        max_iter=300,
        n_init=3
    )
    gmm.fit(X)

    responsibilities = gmm.predict_proba(X) # probability that embedding i belongs to component k
    labels = responsibilities.argmax(axis=1) # most likely component for embedding

    # cluster using hard labels
    # maps cluster # -> list of entities
    clusters:dict[int, Cluster] = {k:[] for k in range(n_components)}
    for i, k in enumerate(labels):
        clusters[int(k)].append(entities[i])

    aggregates:list[AggEntity] = []
    cluster_backref:dict[AggEntity, Cluster] = {}
    for _, cluster in clusters.items():
        new_parent:AggEntity = await build_aggregate_entity(cluster, findings_map)
        aggregates.append(new_parent)
        cluster_backref[new_parent] = cluster
        # insert it into graph at next layer, creating an :IS_CHILD_OF relation between all children -> the new parent
        await mg_driver.create_aggregate_entity(new_parent, cluster, layer+1)

    allowed_tokens = ( max_depth - layer ) * 40 * 2 # (TAU) from LeanRAG
    # 4.all cluster aggregates are inserted, now create inter-cluster relations between (complete subgraph)
    for cj, ck in combinations(aggregates, 2):
        # collect all inter-cluster relations between entities in cj and entities in ck
        inter_cluster_rel:list[IntrClusterRel] = await mg_driver.get_inter_cluster_relations(cluster_backref[cj],cluster_backref[ck])

        # cumulative number of tokens for all inter-cluster relation descriptions.. LeanRAG 'connectivity strength'
        n_tokens_intercluster_rel = sum([len(Tokenizer.encode(r['desc'])) for r in inter_cluster_rel])

        if n_tokens_intercluster_rel > allowed_tokens:
            icr_desc:str = signatures.generate_aggregate_rel_desc(cj, ck, inter_cluster_rel)
        else:
            icr_desc:str = icr_desc_fallback_concat(inter_cluster_rel)

        # create relation in memgraph (1 layer up)
        await mg_driver.create_inter_cluster_relation(cj,ck, icr_desc, layer+1)
    
    # recurse
    await aggregate_layer(layer+1)

await mg_driver.init()
total_entities = await mg_driver.count_entities()
max_depth = round(log(len(total_entities), CLUSTER_SIZE)) + 1 # from LeanRAG
await aggregate_layer(0)